In [ ]:
pip install transformers datasets peft accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [ ]:
pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 41.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


In [ ]:

import os
import math
import time
import random
import json
from dataclasses import dataclass
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
import evaluate
import numpy as np
from peft import LoraConfig, get_peft_model

In [ ]:
BASE_MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "dair-ai/emotion"
MAX_LENGTH = 128
TRAIN_SUBSET_SIZE = 3000
NUM_EPOCHS = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

accuracy_metric = evaluate.load("accuracy")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print("Loading dataset:")
raw_datasets = load_dataset(DATASET_NAME)

train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))
val_dataset = raw_datasets["validation"]
test_dataset = raw_datasets["test"]

print("Train subset size:", len(train_dataset))
print("Validation size:", len(val_dataset))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

print("Tokenising:")
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_val = tokenized_val.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

tokenized_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)
tokenized_val.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)
tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

num_labels = raw_datasets["train"].features["label"].num_classes
id2label = {i: l for i, l in enumerate(raw_datasets["train"].features["label"].names)}
label2id = {v: k for k, v in id2label.items()}

Loading dataset:


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenising:


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# compute metrics for trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

In [ ]:
import math
import time
import json
import random
from dataclasses import dataclass
import numpy as np
import torch

SEARCH_SPACE = {
    "learning_rate": {
        "type": "log_uniform",
        "min": 1e-6,
        "max": 5e-4,
    },
    "warmup_ratio": {
        "type": "continuous",
        "min": 0.0,
        "max": 0.2,
    },
    "lora_r": {
        "type": "discrete",
        "values": [2, 4, 8, 16, 32],
    },
    "lora_alpha": {
        "type": "discrete",
        "values": [8, 16, 32, 64, 128],
    },
    "lora_dropout": {
        "type": "continuous",
        "min": 0.0,
        "max": 0.3,
    },
    "target_modules": {
        "type": "categorical",
        "values": [
            "attention-only",
            "attention-plus-ffn",
        ],
    },
}

@dataclass
class HyperParams:
  learning_rate: float
  warmup_ratio: float
  lora_r: int
  lora_alpha: int
  lora_dropout: float
  target_modules_option: str

def decode_position_to_hparams(position: np.ndarray) -> HyperParams:
  x = np.clip(np.asarray(position, dtype=float), 0.0, 1.0)

  # learning_rate
  lr_cfg = SEARCH_SPACE["learning_rate"]
  lr_min = lr_cfg["min"]
  lr_max = lr_cfg["max"]
  log_min = math.log(lr_min)
  log_max = math.log(lr_max)
  log_lr = log_min + x[0] * (log_max - log_min)
  learning_rate = math.exp(log_lr)

  # warmup_ratio
  w_cfg = SEARCH_SPACE["warmup_ratio"]
  warmup_ratio = w_cfg["min"] + x[1] * (w_cfg["max"] - w_cfg["min"])

  def pick_discrete(val: float, choices: list[int]) -> int:
    idx = int(round(val * (len(choices) - 1)))
    idx = max(0, min(len(choices) - 1, idx))
    return choices[idx]

  # lora rank
  r_choices = SEARCH_SPACE["lora_r"]["values"]
  lora_r = pick_discrete(x[2], r_choices)

  # lora alpha
  alpha_choices = SEARCH_SPACE["lora_alpha"]["values"]
  lora_alpha = pick_discrete(x[3], alpha_choices)

  # lora dropout
  d_cfg = SEARCH_SPACE["lora_dropout"]
  lora_dropout = d_cfg["min"] + x[4] * (d_cfg["max"] - d_cfg["min"])

  # target modules option
  t_choices = SEARCH_SPACE["target_modules"]["values"]
  t_idx = 0 if x[5] < 0.5 else 1
  target_modules_option = t_choices[t_idx]

  return HyperParams(
      learning_rate=learning_rate,
      warmup_ratio=warmup_ratio,
      lora_r=lora_r,
      lora_alpha=lora_alpha,
      lora_dropout=lora_dropout,
      target_modules_option=target_modules_option,
      )

def get_target_modules_from_option(option: str) -> list[str]:
  if option == "attention-only":
    return ["q_lin", "v_lin"]
  elif option == "attention-plus-ffn":
    return ["q_lin", "v_lin", "ffn.lin1", "ffn.lin2"]
  else:
    raise ValueError(f"Unknown target_modules_option: {option}")


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from transformers import set_seed

def count_trainable_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate_config_multi(
    hparams: HyperParams,
    seed: int,
    run_name: str,
) -> tuple[float, int, float, float]:

    set_seed(seed)
    start_time = time.time()

    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    # LoRA configuration
    target_modules = get_target_modules_from_option(hparams.target_modules_option)
    lora_config = LoraConfig(
        r=hparams.lora_r,
        lora_alpha=hparams.lora_alpha,
        lora_dropout=hparams.lora_dropout,
        target_modules=target_modules,
        bias="none",
        task_type="SEQ_CLS",
    )
    model = get_peft_model(base_model, lora_config)
    model.to(DEVICE)

    num_trainable = count_trainable_parameters(model)

    training_args = TrainingArguments(
        output_dir=f"./tmp/{run_name}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=hparams.learning_rate,
        warmup_ratio=hparams.warmup_ratio,
        weight_decay=0.01,
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        load_best_model_at_end=False,
        remove_unused_columns=True,
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_metrics = trainer.evaluate(eval_dataset=tokenized_val)
    val_acc = float(eval_metrics.get("eval_accuracy", float("nan")))
    error = 1.0 - val_acc

    train_time = time.time() - start_time

    print(
        f"[{run_name}] seed={seed} | val_acc={val_acc:.4f} "
        f"(error={error:.4f}), trainable_params={num_trainable}, "
        f"time={train_time:.1f}s"
    )

    del trainer
    del model
    del base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return error, num_trainable, val_acc, train_time


def evaluate_position_multi(
    position: list[float] | np.ndarray,
    seed: int,
    eval_index: int,
    run_prefix: str,
) -> tuple[float, int, float, HyperParams, float]:
    hparams = decode_position_to_hparams(np.array(position))
    run_name = f"{run_prefix}_eval{eval_index}_seed{seed}"

    error, num_params, val_acc, train_time = evaluate_config_multi(
        hparams=hparams,
        seed=seed,
        run_name=run_name,
    )
    return error, num_params, val_acc, hparams, train_time

In [ ]:
def get_pareto_front(solutions: list[dict]) -> list[dict]:
    pareto = []
    for i, si in enumerate(solutions):
        ei = si["error"]
        pi = si["num_params"]
        dominated = False

        for j, sj in enumerate(solutions):
            if i == j:
                continue
            ej = sj["error"]
            pj = sj["num_params"]

            if (ej <= ei and pj <= pi) and (ej < ei or pj < pi):
                dominated = True
                break

        if not dominated:
            pareto.append(si)

    return pareto


In [ ]:
def get_gpu_info():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        mem_gb = round(props.total_memory / (1024**3), 2)
        return name, mem_gb
    else:
        return "CPU", 0.0


def scalarised_value(
    error_value: float,
    params_value: int,
    weight_vector: tuple[float, float],
    ideal_error_value: float,
    ideal_params_value: float,
) -> float:

    w1, w2 = weight_vector
    term1 = w1 * abs(error_value - ideal_error_value)
    term2 = w2 * abs(params_value - ideal_params_value)
    return max(term1, term2)


def moead_de_for_one_seed(
    seed: int,
    pop_size: int = 10,
    num_generations: int = 5,
    T: int = 3,
    F: float = 0.5,
    CR: float = 0.9,
    run_prefix: str = "moead_de",
) -> dict:

    print(f"\n MOEA/D-DE (seed = {seed}) | pop_size = {pop_size}, generation = {num_generations}, neighbourhood size (T) = {T}")

    random.seed(seed)
    np.random.seed(seed)

    dim = 6

    # Weight vectors for the 2 objectives (val_error, training_params)
    weight_vectors: list[tuple[float, float]] = []
    if pop_size == 1:
        weight_vectors.append((0.5, 0.5))
    else:
        for i in range(pop_size):
            w1 = i / (pop_size - 1)
            w2 = 1.0 - w1
            weight_vectors.append((w1, w2))

    # Neighbourhoods based on Euclidean distance
    neighbourhoods: list[list[int]] = []
    for i in range(pop_size):
        distances = []
        wi1, wi2 = weight_vectors[i]
        for j in range(pop_size):
            wj1, wj2 = weight_vectors[j]
            dist = math.sqrt((wi1 - wj1) ** 2 + (wi2 - wj2) ** 2)
            distances.append((dist, j))
        distances.sort(key=lambda pair: pair[0])
        neighbours = [idx for (_, idx) in distances[:T]]
        neighbourhoods.append(neighbours)

    #  Initial population
    population: list[list[float]] = []
    for _ in range(pop_size):
        position = [random.random() for _ in range(dim)]
        population.append(position)

    eval_counter = 0
    objectives: list[tuple[float, int]] = []
    trails: list[dict] = []
    all_accuracies: list[float] = []
    all_times: list[float] = []

    ideal_error = float("inf")
    ideal_params = float("inf")

    best_acc_overall = -1.0
    best_trial_no = None

    gpu_name, gpu_mem_gb = get_gpu_info()

    print("\n Initial evaluation (generation 0) ")

    # initial population evaluation
    for i, position in enumerate(population):
        eval_counter += 1
        error, num_params, val_acc, hparams, elapsed_time = evaluate_position_multi(
            position=position,
            seed=seed,
            eval_index=eval_counter,
            run_prefix=f"{run_prefix}_seed{seed}",
        )

        objectives.append((error, num_params))
        all_accuracies.append(val_acc)
        all_times.append(elapsed_time)

        # updating ideal point
        if error < ideal_error:
            ideal_error = error
        if num_params < ideal_params:
            ideal_params = num_params

        # updating global best
        if val_acc > best_acc_overall:
            best_acc_overall = val_acc
            best_trial_no = eval_counter

        trails.append({
            "trial_no": eval_counter,
            "generation": 0,
            "subproblem": i,
            "gpu_memory_gb": gpu_mem_gb,
            "error": float(error),
            "val_acc": float(val_acc),
            "best_val_acc_so_far": float(best_acc_overall),
            "num_params": int(num_params),
            "train_time_sec": float(elapsed_time),
            "hyperparams": {
                "learning_rate": hparams.learning_rate,
                "warmup_ratio": hparams.warmup_ratio,
                "lora_r": hparams.lora_r,
                "lora_alpha": hparams.lora_alpha,
                "lora_dropout": hparams.lora_dropout,
                "target_modules_option": hparams.target_modules_option,
            },
        })

        print(
            f"Init subproblem {i}: acc={val_acc:.4f}, "
            f"error={error:.4f}, params={num_params}"
        )

    # scalar values for generation 0
    scalar_values = []
    for i in range(pop_size):
        error_i, params_i = objectives[i]
        g_val = scalarised_value(
            error_i,
            params_i,
            weight_vectors[i],
            ideal_error,
            ideal_params,
        )
        scalar_values.append(g_val)

    # Main MOEA/D-DE loop over generations
    for gen in range(1, num_generations + 1):
        print(f"\n Generation {gen}/{num_generations} ")

        for i in range(pop_size):
            neighbour_indices = neighbourhoods[i]

            if len(neighbour_indices) < 3:
                candidate_indices = list(range(pop_size))
            else:
                candidate_indices = neighbour_indices

            a_idx, b_idx, c_idx = random.sample(candidate_indices, 3)
            a = population[a_idx]
            b = population[b_idx]
            c = population[c_idx]

            # DE mutation
            mutant = []
            for d in range(dim):
                val = a[d] + F * (b[d] - c[d])
                val = max(0.0, min(1.0, val))
                mutant.append(val)

            # DE crossover
            target = population[i]
            offspring = []
            j_rand = random.randint(0, dim - 1)
            for d in range(dim):
                if random.random() < CR or d == j_rand:
                    offspring.append(mutant[d])
                else:
                    offspring.append(target[d])

            # offspring evaluation
            eval_counter += 1
            new_error, new_params, new_val_acc, new_hparams, new_elapsed = evaluate_position_multi(
                position=offspring,
                seed=seed,
                eval_index=eval_counter,
                run_prefix=f"{run_prefix}_seed{seed}",
            )

            all_accuracies.append(new_val_acc)
            all_times.append(new_elapsed)

            if new_error < ideal_error:
                ideal_error = new_error
            if new_params < ideal_params:
                ideal_params = new_params

            if new_val_acc > best_acc_overall:
                best_acc_overall = new_val_acc
                best_trial_no = eval_counter

            trails.append({
                "trial_no": eval_counter,
                "generation": gen,
                "subproblem": i,
                "gpu_memory_gb": gpu_mem_gb,
                "error": float(new_error),
                "val_acc": float(new_val_acc),
                "best_val_acc_so_far": float(best_acc_overall),
                "num_params": int(new_params),
                "train_time_sec": float(new_elapsed),
                "hyperparams": {
                    "learning_rate": new_hparams.learning_rate,
                    "warmup_ratio": new_hparams.warmup_ratio,
                    "lora_r": new_hparams.lora_r,
                    "lora_alpha": new_hparams.lora_alpha,
                    "lora_dropout": new_hparams.lora_dropout,
                    "target_modules_option": new_hparams.target_modules_option,
                },
            })

            # MOEA/D-DE replacement in neighbourhood
            for j in neighbour_indices:
                current_error, current_params = objectives[j]
                g_old = scalarised_value(
                    current_error,
                    current_params,
                    weight_vectors[j],
                    ideal_error,
                    ideal_params,
                )
                g_new = scalarised_value(
                    new_error,
                    new_params,
                    weight_vectors[j],
                    ideal_error,
                    ideal_params,
                )
                if g_new <= g_old:
                    population[j] = offspring[:]
                    objectives[j] = (new_error, new_params)

            print(
                f"Gen {gen}, subproblem {i}: acc={new_val_acc:.4f}, "
                f"error={new_error:.4f}, params={new_params}"
            )

    # Final solutions and per-seed Pareto front
    final_solutions = []
    for position, (err, nparams) in zip(population, objectives):
        hp = decode_position_to_hparams(np.array(position))
        acc = 1.0 - err
        final_solutions.append({
            "error": float(err),
            "val_accuracy": float(acc),
            "num_params": int(nparams),
            "hyperparams": {
                "learning_rate": hp.learning_rate,
                "warmup_ratio": hp.warmup_ratio,
                "lora_r": hp.lora_r,
                "lora_alpha": hp.lora_alpha,
                "lora_dropout": hp.lora_dropout,
                "target_modules_option": hp.target_modules_option,
            },
        })

    pareto_front = get_pareto_front(final_solutions)

    val_acc_mean = float(np.mean(all_accuracies))
    val_acc_std = float(np.std(all_accuracies))
    avg_time = float(np.mean(all_times))
    total_time = float(np.sum(all_times))

    return {
        "seed": seed,
        "trails": trails,
        "Best_trail": best_trial_no,
        "Best Accuracy": float(best_acc_overall),
        "val_acc_mean": val_acc_mean,
        "val_acc_std": val_acc_std,
        "avg_train_time_sec": avg_time,
        "total_train_time": total_time,
        "pareto_front": pareto_front,
    }


In [ ]:
def run_moead_de_multi_seed(
    seeds: list[int] = [1, 42, 999, 1234],
    pop_size: int = 10,
    num_generations: int = 5,
    T: int = 3,
    F: float = 0.5,
    CR: float = 0.9,
    json_path: str = "moead_de_results_multi_seed.json",
):
    gpu_name, gpu_mem_gb = get_gpu_info()

    seeds_results = []
    all_val_accs_all_trials = []
    seed_best_accs = []

    for seed in seeds:
        seed_result = moead_de_for_one_seed(
            seed=seed,
            pop_size=pop_size,
            num_generations=num_generations,
            T=T,
            F=F,
            CR=CR,
            run_prefix="moead_de",
        )
        seeds_results.append(seed_result)

        # Overall summary statistics
        for tr in seed_result["trails"]:
            all_val_accs_all_trials.append(tr["val_acc"])
        seed_best_accs.append(seed_result["Best Accuracy"])

    # Overall statistics across seeds
    global_best_seed = None
    global_best_trial_no = None
    global_best_val_acc = -1.0
    global_best_hparams = None

    total_train_time_all = 0.0

    for seed_result in seeds_results:
        total_train_time_all += seed_result["total_train_time"]
        if seed_result["Best Accuracy"] > global_best_val_acc:
            global_best_val_acc = seed_result["Best Accuracy"]
            global_best_seed = seed_result["seed"]
            global_best_trial_no = seed_result["Best_trail"]

            for tr in seed_result["trails"]:
                if tr["trial_no"] == global_best_trial_no:
                    global_best_hparams = tr["hyperparams"]
                    break

    val_acc_mean_over_all = float(np.mean(all_val_accs_all_trials))
    val_acc_std_over_all = float(np.std(all_val_accs_all_trials))
    val_acc_mean_seed_bests = float(np.mean(seed_best_accs))
    val_acc_std_seed_bests = float(np.std(seed_best_accs))

    num_epochs_per_trial = NUM_EPOCHS
    total_trials_per_seed = pop_size * (num_generations + 1)

    formatted_seeds_results = []
    for sr in seeds_results:
        formatted_seeds_results.append({
            "seed": sr["seed"],
            "trails": sr["trails"],
            "Best_trail": sr["Best_trail"],
            "Best Accuracy": sr["Best Accuracy"],
            "val_acc_mean": sr["val_acc_mean"],
            "val_acc_std": sr["val_acc_std"],
            "avg_train_time_sec": sr["avg_train_time_sec"],
            "total_train_time": sr["total_train_time"],
            "pareto_front": sr["pareto_front"],
        })

    results = [
        {
            "method": "moead_de",
            "gpu_name": gpu_name,
            "num_seeds": len(seeds),
            "seeds": seeds,
            "total_trials_per_seed": total_trials_per_seed,
            "num_epochs_per_trial": num_epochs_per_trial,
            "search_space": SEARCH_SPACE,
            "seeds_results": formatted_seeds_results,
            "overall_stats": {
                "global_best_seed": global_best_seed,
                "global_best_trial_no": global_best_trial_no,
                "global_best_val_acc": float(global_best_val_acc),
                "global_best_hyperparams": global_best_hparams,
                "val_acc_mean_over_all_trials": val_acc_mean_over_all,
                "val_acc_std_over_all_trials": val_acc_std_over_all,
                "val_acc_mean_of_seed_bests": val_acc_mean_seed_bests,
                "val_acc_std_of_seed_bests": val_acc_std_seed_bests,
                "total_train_time_sec_over_all_seeds": float(total_train_time_all),
            },
        }
    ]

    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n Saved MOEA/D-DE multi-seed results to: {json_path}")
    return results


In [ ]:
if __name__ == "__main__":
    results = run_moead_de_multi_seed(
        seeds=[1, 42, 999, 1234],
        pop_size=5,
        num_generations=3,
        T=2,
        F=0.5,
        CR=0.9,
        json_path="MOEAD_DE_results.json",
)



 MOEA/D-DE (seed = 1) | pop_size = 5, generation = 3, neighbourhood size (T) = 2

 Initial evaluation (generation 0) 


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1281400033.py:57: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,1.791400
100,1.775500
150,1.764200
200,1.745900
250,1.725200
300,1.715300
350,1.712400
400,1.696200
450,1.694400
500,1.692500


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[moead_de_seed1234_eval19_seed1234] seed=1234 | val_acc=0.4100 (error=0.5900), trainable_params=632070, time=76.3s
Gen 3, subproblem 3: acc=0.4100, error=0.5900, params=632070


/tmp/ipython-input-1281400033.py:57: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,1.809000
100,1.784700
150,1.762800
200,1.737000
250,1.731600
300,1.715600
350,1.692500
400,1.690900
450,1.685500
500,1.668000


[moead_de_seed1234_eval20_seed1234] seed=1234 | val_acc=0.4100 (error=0.5900), trainable_params=632070, time=76.3s
Gen 3, subproblem 4: acc=0.4100, error=0.5900, params=632070

 Saved MOEA/D-DE multi-seed results to: MOEAD_DE_results.json
